# 03 – Weather Download

**Purpose:** Get daily weather for every store city in scope.

**Source:** Open-Meteo Historical Weather API (free, no key)
**Output:** `data/raw/weather_daily.csv`: 9 cities × 1,233 days = 11,097 rows (max/min/mean temperature, precipitation)

**Limitations:**
- Reanalysis data at city-grid level, not exact store location
- Actual weather used; production would use weather forecasts (known in advance)

In [1]:
import requests                      # library for calling web APIs (already installed)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {                           # a dictionary: name → value pairs
    "latitude": -0.18,               # Quito
    "longitude": -78.47,
    "start_date": "2017-08-01",
    "end_date": "2017-08-07",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/Guayaquil"  # Ecuador's local time zone
}

response = requests.get(url, params=params)   # send the request
data = response.json()                         # turn the JSON answer into Python
data["daily"]                                  # show the daily part

{'time': ['2017-08-01',
  '2017-08-02',
  '2017-08-03',
  '2017-08-04',
  '2017-08-05',
  '2017-08-06',
  '2017-08-07'],
 'temperature_2m_max': [20.0, 20.2, 20.1, 19.8, 18.6, 18.0, 17.7],
 'temperature_2m_min': [5.0, 8.4, 6.0, 6.8, 7.7, 6.3, 6.8],
 'precipitation_sum': [0.0, 1.3, 1.0, 0.5, 0.0, 0.0, 0.0]}

In [2]:
import requests
import pandas as pd
import time

# 1. City → (latitude, longitude). Names must match stores.city exactly.
cities = {
    "Quito":      (-0.18, -78.47),
    "Cuenca":     (-2.90, -79.00),
    "Ambato":     (-1.25, -78.62),
    "Loja":       (-4.00, -79.20),
    "Cayambe":    ( 0.04, -78.14),
    "Guayaquil":  (-2.19, -79.89),
    "Machala":    (-3.26, -79.96),
    "Esmeraldas": ( 0.96, -79.65),
    "Daule":      (-1.86, -79.98),
}

url = "https://archive-api.open-meteo.com/v1/archive"
all_cities = []                                   # empty list to collect each city's table

# 2. Loop: same request, once per city
for city, (lat, lon) in cities.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2014-04-01",
        "end_date": "2017-08-15",
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum",
        "timezone": "America/Guayaquil",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()                   # stop with an error if the call failed
    df = pd.DataFrame(response.json()["daily"])   # dictionary of lists → table
    df["city"] = city                             # add a city column
    all_cities.append(df)                         # add this city's table to the list
    print(city, len(df))                          # progress: city name and row count
    time.sleep(1)                                 # pause 1 second between calls (polite)

# 3. Stack all cities into one table and save it
weather = pd.concat(all_cities, ignore_index=True)
weather.to_csv("../data/raw/weather_daily.csv", index=False)
weather.shape

Quito 1233
Cuenca 1233
Ambato 1233
Loja 1233
Cayambe 1233
Guayaquil 1233
Machala 1233
Esmeraldas 1233
Daule 1233


(11097, 6)